In [ ]:
!pip uninstall -y transformers huggingface_hub accelerate peft sentence-transformers
!pip cache purge

!pip install \
transformers==4.36.2 \
huggingface_hub==0.20.3 \
accelerate==0.25.0 \
sentence-transformers==2.2.2 \
peft==0.7.1 \
faiss-cpu

Found existing installation: transformers 4.36.2
Uninstalling transformers-4.36.2:
  Successfully uninstalled transformers-4.36.2
Found existing installation: huggingface-hub 0.20.3
Uninstalling huggingface-hub-0.20.3:
  Successfully uninstalled huggingface-hub-0.20.3
Found existing installation: accelerate 0.25.0
Uninstalling accelerate-0.25.0:
  Successfully uninstalled accelerate-0.25.0
Found existing installation: peft 0.7.1
Uninstalling peft-0.7.1:
  Successfully uninstalled peft-0.7.1
Found existing installation: sentence-transformers 2.2.2
Uninstalling sentence-transformers-2.2.2:
  Successfully uninstalled sentence-transformers-2.2.2
Files removed: 41
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 11.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 98.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.1/33

In [ ]:
!nvidia-smi

!pip install -q openai-whisper
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q ffmpeg-python
!pip install -q torch torchvision torchaudio
!pip install -q transformers timm pillow opencv-python sentencepiece
!apt-get update
!apt-get install -y ffmpeg


Sun Apr  5 15:11:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import os

os.makedirs("video", exist_ok=True)
os.makedirs("frames", exist_ok=True)
os.makedirs("transcripts", exist_ok=True)
os.makedirs("embeddings", exist_ok=True)
os.makedirs("clips", exist_ok=True)

print("Folders created")


Folders created


In [ ]:
from google.colab import files

uploaded = files.upload()

for name in uploaded:
    os.rename(name, "video/input.mp4")


print("Video saved as video/input.mp4")


Saving Alchemy__History_of_Science_10_-_English.mp4 to Alchemy__History_of_Science_10_-_English.mp4
Video saved as video/input.mp4


In [ ]:
import whisper
import json

model = whisper.load_model("base")

result = model.transcribe(
    "video/input.mp4",
    word_timestamps=True
)

segments = []
for seg in result["segments"]:
    segments.append({
        "text": seg["text"].strip(),
        "start": seg["start"],
        "end": seg["end"]
    })

with open("transcripts/transcript.json", "w") as f:
    json.dump(segments, f, indent=2)

print("Transcription completed")
print("Total segments:", len(segments))


100%|███████████████████████████████████████| 139M/139M [00:01<00:00, 78.5MiB/s]


Transcription completed
Total segments: 186


In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [seg["text"] for seg in segments]
text_embeddings = embed_model.encode(texts, convert_to_numpy=True)

dim = text_embeddings.shape[1]
text_index = faiss.IndexFlatL2(dim)
text_index.add(text_embeddings)

faiss.write_index(text_index, "embeddings/text.index")

print("Text FAISS index created")


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


.gitattributes: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

train_script.py: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Text FAISS index created


In [ ]:
print("\n================ TEXT EMBEDDINGS PROOF ================")

print("Total text embeddings in FAISS:", text_index.ntotal)
print("Embedding dimension:", text_index.d)

print("\n--- Sample Text Embeddings (First 3) ---")
for i in range(5):
    print(f"\nSegment {i+1}")
    print("Text:", segments[i]["text"])
    print("Timestamp:", segments[i]["start"], "-", segments[i]["end"])
    print("Embedding (first 10 values):", text_embeddings[i][:10])



================ TEXT EMBEDDINGS PROOF ================
Total text embeddings in FAISS: 104
Embedding dimension: 384

--- Sample Text Embeddings (First 3) ---

Segment 1
Text: Dictionaries are mappings from key objects to value objects.
Timestamp: 6.32 - 10.68
Embedding (first 10 values): [ 0.02082041  0.02374364 -0.0366783  -0.04623299 -0.05538652 -0.08429805
  0.0438313  -0.03299936  0.02518563 -0.01523307]

Segment 2
Text: Dictionaries consists of key value pairs where the keys must be immutable and the values
Timestamp: 11.38 - 17.9
Embedding (first 10 values): [-0.00822609  0.06039197 -0.04190568 -0.07601596 -0.04664455 -0.03410438
  0.05977626 -0.0607768   0.03750325 -0.01687448]

Segment 3
Text: can be anything.
Timestamp: 17.9 - 19.16
Embedding (first 10 values): [-0.06767351  0.03342731 -0.05190263 -0.01629156  0.01996485  0.01717231
  0.01884446 -0.07893151 -0.061715    0.01193847]

Segment 4
Text: Dictionaries themselves are immutable, so this means once you've created your

In [ ]:
import cv2

cap = cv2.VideoCapture("video/input.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)

frame_interval = int(fps * 2)  # 1 frame every 2 seconds
frame_id = 0
saved = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    if frame_id % frame_interval == 0:
        timestamp = frame_id / fps
        cv2.imwrite(f"frames/frame_{timestamp:.2f}.jpg", frame)
        saved += 1

    frame_id += 1

cap.release()
print(f"Extracted {saved} frames")


Extracted 250 frames


In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import torch
import os

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to(device)

visual_segments = []

for img in sorted(os.listdir("frames")):
    image = Image.open(f"frames/{img}").convert("RGB")
    inputs = processor(image, return_tensors="pt").to(device)
    out = blip_model.generate(**inputs, max_new_tokens=30)
    caption = processor.decode(out[0], skip_special_tokens=True)

    timestamp = float(img.split("_")[1].replace(".jpg", ""))
    visual_segments.append({
        "text": caption,
        "start": timestamp,
        "end": timestamp + 1
    })

with open("transcripts/visual_segments.json", "w") as f:
    json.dump(visual_segments, f, indent=2)

print("Visual captions generated:", len(visual_segments))


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Visual captions generated: 393


In [ ]:
visual_texts = [v["text"] for v in visual_segments]
visual_embeddings = embed_model.encode(visual_texts, convert_to_numpy=True)

visual_index = faiss.IndexFlatL2(dim)
visual_index.add(visual_embeddings)

faiss.write_index(visual_index, "embeddings/visual.index")

print("Visual FAISS index created")


Visual FAISS index created


In [ ]:
print("\n================ VISUAL EMBEDDINGS PROOF ================")

print("Total visual embeddings in FAISS:", visual_index.ntotal)
print("Embedding dimension:", visual_index.d)

print("\n--- Sample Visual Embeddings (First 3) ---")
for i in range(5):
    print(f"\nVisual Segment {i+1}")
    print("Caption:", visual_segments[i]["text"])
    print("Timestamp:", visual_segments[i]["start"], "-", visual_segments[i]["end"])
    print("Embedding (first 10 values):", visual_embeddings[i][:10])



================ VISUAL EMBEDDINGS PROOF ================
Total visual embeddings in FAISS: 250
Embedding dimension: 384

--- Sample Visual Embeddings (First 3) ---

Visual Segment 1
Caption: a black background with a white and red flower
Timestamp: 0.0 - 1.0
Embedding (first 10 values): [-0.00461409  0.05446748 -0.05593341  0.03795651  0.06155476  0.07953602
  0.04751664 -0.04903771  0.02919584 -0.0686134 ]

Visual Segment 2
Caption: the logo for harvard
Timestamp: 1.96 - 2.96
Embedding (first 10 values): [-0.0140974   0.07579283  0.04445346 -0.07724282  0.04036755 -0.0053834
  0.04552744 -0.0050162   0.08314338 -0.01456951]

Visual Segment 3
Caption: a person drawing a picture on a whiteboard
Timestamp: 101.94 - 102.94
Embedding (first 10 values): [ 0.04583171 -0.00051857 -0.06726023 -0.06830972 -0.03569708  0.04129105
  0.05243862  0.06455833  0.05536192 -0.01221438]

Visual Segment 4
Caption: a person drawing on a whiteboard with a laptop
Timestamp: 103.9 - 104.9
Embedding (first 

In [ ]:
print("\n================ COMBINED VECTOR DATABASE (AUDIO + VISUAL) ================")

combined_db = []

# Add AUDIO / TEXT embeddings
for i in range(len(segments)):
    combined_db.append({
        "modality": "Audio",
        "text": segments[i]["text"],
        "start": segments[i]["start"],
        "end": segments[i]["end"],
        "embedding": text_embeddings[i]
    })

# Add VISUAL embeddings
for i in range(len(visual_segments)):
    combined_db.append({
        "modality": "Visual",
        "text": visual_segments[i]["text"],
        "start": visual_segments[i]["start"],
        "end": visual_segments[i]["end"],
        "embedding": visual_embeddings[i]
    })

print(f"Total entries in combined vector DB: {len(combined_db)}")

print("\n--- SAMPLE COMBINED ENTRIES (First 5) ---")
for i in range(min(10, len(combined_db))):
    item = combined_db[i]
    print(f"\nEntry {i+1}")
    print("Modality:", item["modality"])
    print("Text/Caption:", item["text"])
    print("Timestamp:", item["start"], "-", item["end"])
    print("Embedding (first 10 values):", item["embedding"][:10])



================ COMBINED VECTOR DATABASE (AUDIO + VISUAL) ================
Total entries in combined vector DB: 354

--- SAMPLE COMBINED ENTRIES (First 5) ---

Entry 1
Modality: Audio
Text/Caption: Dictionaries are mappings from key objects to value objects.
Timestamp: 6.32 - 10.68
Embedding (first 10 values): [ 0.02082046  0.02374367 -0.03667834 -0.04623296 -0.05538654 -0.08429803
  0.04383134 -0.03299938  0.02518561 -0.01523307]

Entry 2
Modality: Audio
Text/Caption: Dictionaries consists of key value pairs where the keys must be immutable and the values
Timestamp: 11.38 - 17.9
Embedding (first 10 values): [-0.0082261   0.06039195 -0.04190566 -0.07601599 -0.04664453 -0.03410437
  0.05977626 -0.06077682  0.03750329 -0.0168745 ]

Entry 3
Modality: Audio
Text/Caption: can be anything.
Timestamp: 17.9 - 19.16
Embedding (first 10 values): [-0.06767355  0.03342733 -0.05190264 -0.01629153  0.01996483  0.01717231
  0.01884447 -0.07893153 -0.061715    0.01193852]

Entry 4
Modality: Audio
Te

In [ ]:
from transformers import pipeline

qa_pipeline = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    max_length=64
)

print("Local LLM loaded")


Local LLM loaded


In [ ]:
# ===============================
# FULL SPEECH SUMMARY (CHUNKED)
# ===============================

def chunk_text(text, chunk_size=500):
    words = text.split()
    for i in range(0, len(words), chunk_size):
        yield " ".join(words[i:i + chunk_size])

# 1. Combine full speech
full_speech = " ".join([seg["text"] for seg in segments])

print("Total words in video speech:", len(full_speech.split()))

# 2. Create chunks
speech_chunks = list(chunk_text(full_speech, chunk_size=400))
print("Total chunks:", len(speech_chunks))

chunk_summaries = []

# 3. Summarize each chunk
for i, chunk in enumerate(speech_chunks):
    prompt = f"""
Summarize the following part of an educational lecture in simple language.
Do not skip ideas.

Text:
{chunk}
"""
    summary = qa_pipeline(prompt)[0]["generated_text"]
    chunk_summaries.append(summary)
    print(f"Chunk {i+1} summarized")

# 4. Combine all summaries
full_summary = "\n\n".join(chunk_summaries)

print("\n================ FULL VIDEO SPEECH SUMMARY ================\n")
print(full_summary)


Token indices sequence length is longer than the specified maximum sequence length for this model (539 > 512). Running this sequence through the model will result in indexing errors


Total words in video speech: 1210
Total chunks: 4
Chunk 1 summarized
Chunk 2 summarized
Chunk 3 summarized
Chunk 4 summarized

================ FULL VIDEO SPEECH SUMMARY ================

We'll use the following syntax to construct a dictionary.

Understand the meaning of the word plus. Understand the meaning of the word equals. Understand the order of the two operations.

Learn how to use view objects to find the key in the dictionary.

The first step in a dictionary is to find the object of a word.


In [ ]:
query = input("Ask your question: ")

query_emb = embed_model.encode([query], convert_to_numpy=True)

# Text retrieval
D_t, I_t = text_index.search(query_emb, k=2)
text_context = " ".join(segments[i]["text"] for i in I_t[0])

# Visual retrieval
D_v, I_v = visual_index.search(query_emb, k=2)

visual_context = " ".join(visual_segments[i]["text"] for i in I_v[0])

combined_context = text_context + " " + visual_context

prompt = f"""
Answer ONLY using the context.
Be very short (one sentence).

Context:
{combined_context}

Question:
{query}
"""

answer = qa_pipeline(prompt)[0]["generated_text"]

print("\n✅ FINAL ANSWER:")
print(answer)


Ask your question: Why are dictionaries used for fast look-ups?

✅ FINAL ANSWER:
for performing very fast lookups on unordered data


In [ ]:
from rouge_score import rouge_scorer

# Use the retrieved transcript segment as reference
reference = combined_context  # already exists in your code
generated = answer            # already exists in your code

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
scores = scorer.score(reference, generated)

print("\n================ EVALUATION METRICS ================")
print(f"ROUGE-1: Precision={scores['rouge1'].precision:.4f}, Recall={scores['rouge1'].recall:.4f}, F1={scores['rouge1'].fmeasure:.4f}")
print(f"ROUGE-2: Precision={scores['rouge2'].precision:.4f}, Recall={scores['rouge2'].recall:.4f}, F1={scores['rouge2'].fmeasure:.4f}")
print(f"ROUGE-L: Precision={scores['rougeL'].precision:.4f}, Recall={scores['rougeL'].recall:.4f}, F1={scores['rougeL'].fmeasure:.4f}")

In [ ]:
import subprocess
from IPython.display import Video, display

best_segment = segments[I_t[0][0]]

start = max(0, best_segment["start"] - 0.9)
end = best_segment["end"] + 0.5

subprocess.run([
    "ffmpeg", "-y",
    "-i", "video/input.mp4",
    "-ss", str(start),
    "-to", str(end),
    "clips/answer_clip.mp4"
])

display(Video("clips/answer_clip.mp4", embed=True))
